# Darcy in 3D: native workspaces and CPU/GPU scaling

This notebook uses the public local/global equation API for anisotropic Darcy in the unit cube. Its default CPU control uses 4 × 4 × 4 macrocells, four Q1 subdivisions per coordinate inside each macrocell and unsplit tensor-Q1 normal traces. CPU/GPU campaigns and all available historical panels remain accessible separately.

$$
\begin{aligned}
K(\boldsymbol x)&=m(\boldsymbol x)D,
& m(\boldsymbol x)&=\exp\left(\prod_{j=1}^{3}\sin(20\pi x_j)\right),\\
p_{\mathrm{exact}}(\boldsymbol x)&=\prod_{j=1}^{3}\sin(\pi x_j),
& f&=-\nabla\cdot(K\nabla p_{\mathrm{exact}}).
\end{aligned}
$$

The tensor $D$ is symmetric positive definite; zero exterior pressure fixes the physical solution without a mean gauge. Local native workspaces reuse geometry, spaces, compiled forms and geometry-only trace/mean blocks. Every cell's material operator and source, and every solver factorization, remain fresh.

$$
\begin{aligned}
a_K(p,v)&=\int_K K\nabla p\cdot\nabla v, &L_K(v)&=\int_Kfv,\\
b_K(\lambda,v)&=\int_{\partial K}\lambda v,
&c_K(p,\mu)&=-\int_{\partial K}p\mu.
\end{aligned}
$$

The custom interface supplies the canonical-to-outward-normal transport. The global multiplier system also retains one local pressure constant. Reported Darcy flux is the broken physical field $-K\nabla p_h$; no fine-cell H(div) property is assumed.

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/459efa13b5376ab66440dde399289c04fe31f411d1222124cf88287760a166a2/darcy_3d_parallel_scalability-companion.zip"
COMPANION_SHA256 = "459efa13b5376ab66440dde399289c04fe31f411d1222124cf88287760a166a2"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("introduction/darcy_3d_parallel_scalability.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import inspect
import numpy as np
from IPython.display import Code, display
from pymhm import MeshHierarchy, bind_interface, bind_problem
from pymhm.core.equations import Equation
from pymhm.core.multiscale import assemble
from pymhm.execution.cpu import ExecutionConfig
from examples.introduction import scaling_3d as study

DATA = study.DarcyData(period=0.1)
FINE_N, MACRO_COUNT = 16, 4
TRACE_DEGREE, ASSEMBLY_QUADRATURE_DEGREE = 1, 8
print({"anisotropy": DATA.anisotropy, "source_control": study.verify_source(DATA)})


These are the actual executed physical-data and local-equation declarations. Native form compilation, workspace ownership and the importable spawn workers live in [scaling_3d.py](https://github.com/ipes-lncc/pymhm/tree/main/examples/introduction/scaling_3d.py). `create_cell_workspace` contains the full UFL volume and face forms; `LocalProvider` owns resource lifetime and orientation.

In [ ]:
display(Code(inspect.getsource(study.symbolic_data), language="python"))
display(Code(inspect.getsource(study.assemble_cell_equations), language="python"))


Bind the hierarchy and the custom normal-trace interface, then assemble and solve through the generic API. A spawned worker imports the provider directly; no cell compilation or temporary Python module is needed.

In [ ]:
macro = study.macro_mesh(MACRO_COUNT)
provider = study.LocalProvider(
    macro, FINE_N // MACRO_COUNT, data=DATA,
    trace_degree=TRACE_DEGREE, quadrature_degree=ASSEMBLY_QUADRATURE_DEGREE,
)
problem = bind_problem(
    MeshHierarchy(macro, provider.local_mesh),
    study.TensorFaceInterface(macro, TRACE_DEGREE), provider,
    global_equation=Equation(0, 0), retained=1,
)
system = assemble(problem, execution=ExecutionConfig(
    "process", 2, native_threads=1, batch_size=2, pipeline=True,
))
solution = system.solve()
print(study.mhm_physical_errors(system, solution, data=DATA, order=7))
print({"trace_dofs": problem.trace_size, "relative_residual": solution.residual})


The default study repeats the original small serial/process and LU/AMG controls, independently assembles conforming Q1 references, checks physical errors and saves the executed basis for replay. Spatial slices compare pressure and vector flux with the exact solution and show the actual macro mesh. Historical timing and accelerator panels retain their own source revisions and hardware.

In [ ]:
RUN_SMALL_REPRODUCTION = True
RUN_LARGE_CAMPAIGN = False
RUN_GPU_COMPONENT = False
RUN_FULL_GPU_WORKFLOW = False
reproduction = study.run_study(
    RUN_SMALL_REPRODUCTION=RUN_SMALL_REPRODUCTION,
    RUN_LARGE_CAMPAIGN=RUN_LARGE_CAMPAIGN,
    RUN_GPU_COMPONENT=RUN_GPU_COMPONENT,
    RUN_FULL_GPU_WORKFLOW=RUN_FULL_GPU_WORKFLOW,
    data=DATA, trace_degree=TRACE_DEGREE,
    quadrature_degree=ASSEMBLY_QUADRATURE_DEGREE, small_fine_n=FINE_N,
)


Set `RUN_LARGE_CAMPAIGN=True` for the complete 64/96/128-axis CPU strong/weak campaign (32 workers at the largest configuration). `RUN_GPU_COMPONENT=True` measures original local condensation on two GPUs; `RUN_FULL_GPU_WORKFLOW=True` includes CPU assembly, dedicated GPU solves, ordered global assembly and reconstruction with both cuDSS and AMGX. Optional device runs require the corresponding locked accelerator environment and two usable devices.

The full helpers preserve all resource checks, synchronization, coefficient/basis archives and physical comparisons. The default run never treats archived timings as measurements of the current execution.